# SNOOPPI data audit and evaluation protocol

**Status:** drafting code for review; no scientific result is claimed.

**When to run:** first, before fitting any new baseline or ablation.

**Inputs:** the existing `splits/train.csv`, `validation.csv`, and
`test.csv` created by notebook 01.

**Outputs:** optional audit tables and one figure under
`tables/drafting_code` and `figures/drafting_code`.

**Safety:** `WRITE_OUTPUTS=False` by default. This notebook never rewrites
the source split CSVs. Sequence values are hashed in exported audits.

The audit asks whether the pair-row split leaks normalized pairs or
proteins, whether normalization/truncation collapses distinct raw inputs,
and whether simple length-only features can predict the label. A strong
length-only model is evidence of sampling bias, not a useful PPI model.

**Shared protocol:** set `SNOOPPI_ROOT` and `SNOOPPI_SPLIT_DIR` before setup.
Use the same split directory in every draft. Existing caches use original raw
sequence keys; this suite does not regenerate embeddings. Metadata uses full
sequence hashes, while leakage audits also check normalized 1,024-residue inputs.


## 1. Setup and visible configuration

In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WRITE_OUTPUTS = False
RUN_TAG = "data_audit_v1"
PLOT_COLORS = {"negative": "#D58632", "positive": "#176B87"}

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})

## 2. Load the exact shared splits

In [ ]:
splits = load_splits(PATHS)
manifest = split_manifest(PATHS)

summary_rows = []
for split_name, frame in splits.items():
    summary_rows.append({
        "split": split_name,
        "pairs": len(frame),
        "positives": int(frame[LABEL].sum()),
        "negatives": int((1 - frame[LABEL]).sum()),
        "positive_prevalence": float(frame[LABEL].mean()),
        "unique_raw_A": frame[SEQUENCE_A].nunique(),
        "unique_raw_B": frame[SEQUENCE_B].nunique(),
    })

split_summary = pd.DataFrame(summary_rows)
display(split_summary)
print("Split SHA-256:")
display(pd.Series(manifest["sha256"], name="sha256").to_frame())
print("Preprocessing contract:", preprocessing_report(splits))


## 3. Hash normalized proteins and unordered pairs

In [ ]:
audited = {name: add_pair_ids(frame) for name, frame in splits.items()}

duplicate_summary = pd.DataFrame([
    {
        "split": name,
        "duplicate_unordered_pairs": int(frame["pair_id"].duplicated().sum()),
        "label_conflicting_pairs": int(
            frame.groupby("pair_id")[LABEL].nunique().gt(1).sum()
        ),
    }
    for name, frame in audited.items()
])
display(duplicate_summary)

## 4. Measure cross-split leakage

Pair overlap is the strictest duplicate check. Protein overlap is expected
in the original pair-row split but directly motivates cold-protein and
cold-target evaluations.

In [ ]:
def id_sets(frame):
    proteins = set(frame["protein_A_id"]) | set(frame["protein_B_id"])
    return set(frame["pair_id"]), proteins

overlap_rows = []
for left_name, right_name in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    left_pairs, left_proteins = id_sets(audited[left_name])
    right_pairs, right_proteins = id_sets(audited[right_name])
    shared_pairs = left_pairs & right_pairs
    left_labels = audited[left_name].groupby("pair_id")[LABEL].agg(set)
    right_labels = audited[right_name].groupby("pair_id")[LABEL].agg(set)
    overlap_rows.append({
        "left": left_name,
        "right": right_name,
        "overlapping_pairs": len(left_pairs & right_pairs),
        "overlapping_pairs_with_conflicting_labels": sum(
            left_labels[pair_id] != right_labels[pair_id] for pair_id in shared_pairs
        ),
        "overlapping_proteins": len(left_proteins & right_proteins),
        "right_pair_overlap_rate": len(left_pairs & right_pairs) / max(1, len(right_pairs)),
        "right_protein_overlap_rate": len(left_proteins & right_proteins) / max(1, len(right_proteins)),
    })

overlap_summary = pd.DataFrame(overlap_rows)
display(overlap_summary)
# ProtT5 has a distinct historical normalization rule; audit its input collisions too.
prott5_duplicate_summary = pd.DataFrame([
    {"split": name, "duplicate_pairs": int(frame.prott5_pair_id.duplicated().sum()),
     "conflicting_pairs": int(frame.groupby("prott5_pair_id")[LABEL].nunique().gt(1).sum())}
    for name, frame in audited.items()
])
prott5_overlap_summary = pd.DataFrame([
    {"left": left, "right": right,
     "overlapping_pairs": len(set(audited[left].prott5_pair_id) & set(audited[right].prott5_pair_id)),
     "overlapping_proteins": len((set(audited[left].prott5_protein_A_id) | set(audited[left].prott5_protein_B_id)) &
                                 (set(audited[right].prott5_protein_A_id) | set(audited[right].prott5_protein_B_id)))}
    for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]
])
display(prott5_duplicate_summary, prott5_overlap_summary)


## 5. Check normalization and 1,024-residue truncation collisions

In [ ]:
raw_sequences = pd.concat(
    [frame[[SEQUENCE_A, SEQUENCE_B]].stack() for frame in splits.values()],
    ignore_index=True,
).astype(str)
sequence_map = pd.DataFrame({
    "raw_sequence": raw_sequences,
    "normalized_id": raw_sequences.map(sequence_id),
    "raw_length": raw_sequences.str.replace(" ", "", regex=False).str.len(),
})

collision_counts = sequence_map.groupby("normalized_id")["raw_sequence"].nunique()
collision_summary = pd.DataFrame({
    "unique_raw_sequences": [sequence_map["raw_sequence"].nunique()],
    "unique_normalized_sequences": [sequence_map["normalized_id"].nunique()],
    "normalized_ids_with_multiple_raw_sequences": [int(collision_counts.gt(1).sum())],
    "raw_sequences_over_1024_residues": [int(sequence_map["raw_length"].gt(1024).sum())],
})
display(collision_summary)

## 5a. Reviewed MMseqs2 cluster overlap

The supplied ProtT5 notebook contains no MMseqs2 step. If a reviewed cluster
mapping exists, check it against the selected CSVs rather than assuming those
CSVs are cluster-disjoint. A cluster assignment is not itself a split manifest.
Use `raw/mmseqs2_clusters.csv`, or set `SNOOPPI_CLUSTER_FILE` to its actual path.
The mapping schema and full-sequence hashing rule are documented in the README.


In [ ]:
from snooppi_protocol import load_reviewed_mapping, mapped_overlap_report
CLUSTER_PATH = Path(os.environ.get("SNOOPPI_CLUSTER_FILE", str(PATHS["clusters"])))
if CLUSTER_PATH.exists():
    cluster_mapping = load_reviewed_mapping(CLUSTER_PATH, "cluster_id", pd.concat(splits.values()))
    cluster_overlap = mapped_overlap_report(splits, cluster_mapping, "cluster_id")
    display(cluster_overlap)
    print("Cluster mapping SHA-256:", file_sha256(CLUSTER_PATH))
else:
    print("MMseqs2 overlap NOT VERIFIED: no reviewed cluster mapping found at", CLUSTER_PATH)


## 6. Length-only shortcut baseline

This deliberately weak model sees no amino-acid identity. It uses only
partner lengths and their relationship. Unexpectedly high test AUPRC/MCC
suggests the label construction may be recoverable from sampling artifacts.

In [ ]:
from snooppi_utils import select_threshold, summarize_predictions

def length_features(frame):
    length_a = frame[SEQUENCE_A].astype(str).str.replace(" ", "", regex=False).str.len()
    length_b = frame[SEQUENCE_B].astype(str).str.replace(" ", "", regex=False).str.len()
    return pd.DataFrame({
        "log_length_sum": np.log1p(length_a) + np.log1p(length_b),
        "absolute_log_length_difference": np.abs(np.log1p(length_a) - np.log1p(length_b)),
        "minimum_log_length": np.minimum(np.log1p(length_a), np.log1p(length_b)),
        "maximum_log_length": np.maximum(np.log1p(length_a), np.log1p(length_b)),
    })

shortcut_model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=44)),
])
# Audit completion must remain possible even when fitting is unsafe.
try:
    assert_model_ready(splits)
except (ValueError, FileNotFoundError) as exc:
    shortcut_metrics = pd.DataFrame([{"model": "length_only_logistic_regression", "status": f"skipped: {exc}"}])
    display(shortcut_metrics)
else:
    shortcut_model.fit(length_features(splits["train"]), splits["train"][LABEL])
    
    validation_probability = shortcut_model.predict_proba(length_features(splits["validation"]))[:, 1]
    threshold = select_threshold(splits["validation"][LABEL].to_numpy(), validation_probability)
    test_probability = shortcut_model.predict_proba(length_features(splits["test"]))[:, 1]
    shortcut_metrics = pd.DataFrame([
        summarize_predictions(splits["test"][LABEL].to_numpy(), test_probability, threshold)
    ])
    shortcut_metrics.insert(0, "model", "length_only_logistic_regression")
    display(shortcut_metrics.round(4))


## 7. Visual audit and optional exports

In [ ]:
plot_frame = split_summary.set_index("split")[["negatives", "positives"]]
axis = plot_frame.plot.bar(
    stacked=True,
    color=[PLOT_COLORS["negative"], PLOT_COLORS["positive"]],
)
axis.set(
    title=f"SNOOPPI class counts: {PATHS['splits'].name}",
    xlabel="Split",
    ylabel="Interaction pairs",
)
axis.tick_params(axis="x", rotation=0)
axis.legend(["Negative", "Positive"], frameon=False, title="Label")
axis.figure.tight_layout()
plt.show()

if WRITE_OUTPUTS:
    output_files = [PATHS["tables"] / f"{RUN_TAG}_{suffix}.csv" for suffix in
                    ["split_summary", "duplicates", "overlap", "normalization_collisions", "shortcut_metrics", "prott5_duplicates", "prott5_overlap"]]
    output_files += [PATHS["figures"] / f"{RUN_TAG}_class_counts.png",
                     PATHS["tables"] / f"{RUN_TAG}_manifest.json"]
    prepare_outputs(output_files)
    save_json(make_run_manifest(PATHS, splits, {"run_tag": RUN_TAG, "audit_only": True}), output_files[-1])
    split_summary.to_csv(PATHS["tables"] / f"{RUN_TAG}_split_summary.csv", index=False)
    duplicate_summary.to_csv(PATHS["tables"] / f"{RUN_TAG}_duplicates.csv", index=False)
    overlap_summary.to_csv(PATHS["tables"] / f"{RUN_TAG}_overlap.csv", index=False)
    collision_summary.to_csv(PATHS["tables"] / f"{RUN_TAG}_normalization_collisions.csv", index=False)
    prott5_duplicate_summary.to_csv(PATHS["tables"] / f"{RUN_TAG}_prott5_duplicates.csv", index=False)
    prott5_overlap_summary.to_csv(PATHS["tables"] / f"{RUN_TAG}_prott5_overlap.csv", index=False)
    shortcut_metrics.to_csv(PATHS["tables"] / f"{RUN_TAG}_shortcut_metrics.csv", index=False)
    axis.figure.savefig(PATHS["figures"] / f"{RUN_TAG}_class_counts.png", dpi=300, bbox_inches="tight")
    print("Wrote audit outputs with run tag:", RUN_TAG)
else:
    print("Preview only. Set WRITE_OUTPUTS=True after reviewing the tables.")

## 8. Interpretation checklist

Before model comparison, record:

- whether normalized unordered pairs cross split boundaries;
- how much protein identity is shared with validation/test;
- whether conflicting labels exist for a canonical pair;
- whether truncation or normalization collapses raw sequences;
- whether length-only performance is materially above prevalence and chance.

Do not describe the current split as cold-target or protein-disjoint unless
the corresponding overlap counts are zero by construction.